**Run 8/12: Sinusoidal PE, seed=456** \
**ImageNet-100 (resized 256px), ViT-Base, 300 epochs**



In [ ]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

# ============================================
# EXTRACT IMAGENET-100 + RESIZE TO 256px
# skips if already done
# ============================================

import os, time, tarfile, shutil, subprocess
from PIL import Image

DRIVE_BASE = '/content/drive/My Drive/pe_experiment'
RESULTS_DIR = os.path.join(DRIVE_BASE, 'results')
SCRIPT_DIR = DRIVE_BASE
DATA_DIR = '/content/imagenet100_resized'
IN100_RAW = '/content/imagenet100_raw'
os.makedirs(RESULTS_DIR, exist_ok=True)

train_path = os.path.join(DATA_DIR, 'train')
val_path = os.path.join(DATA_DIR, 'val')

if (os.path.exists(train_path) and len(os.listdir(train_path)) >= 100
    and os.path.exists(val_path) and len(os.listdir(val_path)) >= 100):
    print(f' ImageNet-100 already on SSD (resized)')
    print(f'  Train: {len(os.listdir(train_path))} classes')
    print(f'  Val: {len(os.listdir(val_path))} classes')
else:
    # --- Find tar files on Drive ---
    imagenet_dir = os.path.join(DRIVE_BASE, 'imagenet')
    TRAIN_TAR = None
    VAL_TAR = None
    for name in ['ILSVRC2012_img_train.tar', 'ILSVRC2012_img_train']:
        c = os.path.join(imagenet_dir, name)
        if os.path.exists(c): TRAIN_TAR = c; break
    for name in ['ILSVRC2012_img_val.tar', 'ILSVRC2012_img_val']:
        c = os.path.join(imagenet_dir, name)
        if os.path.exists(c): VAL_TAR = c; break
    print(f'Train tar: {TRAIN_TAR}')
    print(f'Val tar: {VAL_TAR}')

    # --- Download class list ---
    subprocess.run(['wget', '-q',
        'https://raw.githubusercontent.com/HobbitLong/CMC/master/imagenet100.txt',
        '-O', '/content/imagenet100.txt'], check=True)
    with open('/content/imagenet100.txt') as f:
        class_set = set(line.strip() for line in f if line.strip())
    print(f'Target classes: {len(class_set)}')

    t0 = time.time()

    # --- Extract train ---
    print(f'\nStep 1/3: Extracting train from tar...')
    raw_train = os.path.join(IN100_RAW, 'train')
    os.makedirs(raw_train, exist_ok=True)
    tf = tarfile.open(TRAIN_TAR, 'r|')
    found = 0
    for member in tf:
        if not member.name.endswith('.tar'):
            continue
        class_name = os.path.basename(member.name).replace('.tar', '')
        if class_name in class_set:
            class_dir = os.path.join(raw_train, class_name)
            os.makedirs(class_dir, exist_ok=True)
            fileobj = tf.extractfile(member)
            with tarfile.open(fileobj=fileobj, mode='r|') as ctf:
                ctf.extractall(class_dir)
            found += 1
            if found % 10 == 0:
                print(f'  {found}/100 classes ({time.time()-t0:.0f}s)')
            if found >= len(class_set):
                break
        else:
            tf.extractfile(member).read()
    tf.close()
    print(f'  Train extracted: {found} classes')

    # --- Extract val ---
    print(f'\nStep 2/3: Extracting val...')
    val_tmp = '/content/val_tmp'
    os.makedirs(val_tmp, exist_ok=True)
    with tarfile.open(VAL_TAR, 'r|') as vtf:
        vtf.extractall(val_tmp)
    subprocess.run(['wget', '-q',
        'https://raw.githubusercontent.com/tensorflow/models/master/research/slim/datasets/imagenet_2012_validation_synset_labels.txt',
        '-O', '/content/val_synsets.txt'], check=True)
    with open('/content/val_synsets.txt') as f:
        val_synsets = [line.strip() for line in f]
    raw_val = os.path.join(IN100_RAW, 'val')
    os.makedirs(raw_val, exist_ok=True)
    val_images = sorted([f for f in os.listdir(val_tmp) if f.endswith('.JPEG')])
    for img_name, synset in zip(val_images, val_synsets):
        if synset in class_set:
            class_dir = os.path.join(raw_val, synset)
            os.makedirs(class_dir, exist_ok=True)
            shutil.move(os.path.join(val_tmp, img_name), os.path.join(class_dir, img_name))
    shutil.rmtree(val_tmp)
    print(f'  Val extracted: {len(os.listdir(raw_val))} classes')

    # --- Resize all images to 256px ---
    print(f'\nStep 3/3: Resizing images to 256px...')
    t_resize = time.time()
    for split in ['train', 'val']:
        src_split = os.path.join(IN100_RAW, split)
        dst_split = os.path.join(DATA_DIR, split)
        for cls in os.listdir(src_split):
            cls_src = os.path.join(src_split, cls)
            cls_dst = os.path.join(dst_split, cls)
            os.makedirs(cls_dst, exist_ok=True)
            for img_name in os.listdir(cls_src):
                dst_path = os.path.join(cls_dst, img_name)
                if not os.path.exists(dst_path):
                    img = Image.open(os.path.join(cls_src, img_name)).convert('RGB')
                    img.thumbnail((256, 256), Image.LANCZOS)
                    img.save(dst_path, 'JPEG', quality=95)
        print(f'  {split} resized ({time.time()-t_resize:.0f}s)')

    # Clean up raw
    shutil.rmtree(IN100_RAW)

    total = time.time() - t0
    print(f'\n ImageNet-100 ready in {total/60:.1f} min')
    print(f'  Train: {len(os.listdir(train_path))} classes')
    print(f'  Val: {len(os.listdir(val_path))} classes')
    !du -sh {DATA_DIR}


In [ ]:
!pip install -q timm tqdm scikit-learn scipy matplotlib


In [ ]:
# Copy experiment script
import shutil
shutil.copy2(os.path.join(SCRIPT_DIR, 'full_scale_experiment.py'), '/content/')
print('Script copied')
!grep 'torch.compile\|persistent_workers\|autocast' /content/full_scale_experiment.py | head -5
print(' Optimized script confirmed')


In [ ]:
# Check for existing results
import os, json
run_dir = os.path.join(RESULTS_DIR, 'sinusoidal_seed456')
final = os.path.join(run_dir, 'final_model.pth')
ckpt = os.path.join(run_dir, 'last_checkpoint.pth')
hist = os.path.join(run_dir, 'training_history.json')
if os.path.exists(final):
    print(' ALREADY COMPLETE — skip this notebook')
elif os.path.exists(hist):
    with open(hist) as f:
        h = json.load(f)
    print(f' Resuming from epoch {len(h["val_acc"])}/300')
elif os.path.exists(ckpt):
    print(' Checkpoint found — will resume')
else:
    print(' Starting fresh')


In [ ]:
import torch
import os

# Aktivacija TF32 za maksimalne performanse na A100/H100
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Kreiranje lokalnog foldera na SSD-u Colaba
# /content je lokalni SSD, mnogo brži od Drive-a
os.makedirs('/content/train_results', exist_ok=True)

print(" TF32 aktiviran. Lokalni folder spreman.")

In [ ]:
# === RUN TRAINING ===
!python /content/full_scale_experiment.py \
    --data_dir /content/imagenet100_resized \
    --output_dir "/content/drive/My Drive/pe_experiment/results" \
    --pe_type sinusoidal \
    --seed 456 \
    --mode train \
    --epochs 300 \
    --batch_size 128 \
    --num_workers 12 \
    --num_classes 100
